Question 1 Projection of a 3D Point

In [ ]:
import numpy as np

# ==========================================
# 1. CAMERA PARAMETERS & INITIALIZATION
# ==========================================
# Camera intrinsic matrix K
f = 480
cx, cy = 320, 270
K = np.array([
    [f, 0, cx],
    [0, f, cy],
    [0, 0, 1]
])

# Extrinsic parameters (Global to Camera transformation)
R_G_C = np.array([
    [0.5363, -0.8440, 0],
    [0.8440, 0.5363, 0],
    [0, 0, 1]
])
t_C_to_G_global = np.array([[-451.2459], [257.0322], [400]])

# Target 3D point in Global Frame
X_G = np.array([[350], [-250], [-35]])

# Ground truth observed image coordinates
obs_u, obs_v = 241.5, 169.0

# ==========================================
# 2. COMPUTATION
# ==========================================
# (a) Calculate Camera Projection Matrix P
# Transform translation vector from world-centric to camera-centric frame
t_camera_frame = -R_G_C @ t_C_to_G_global
Rt_matrix = np.hstack((R_G_C, t_camera_frame))
P = K @ Rt_matrix

print("--- Question 1(a) ---")
print("Camera Projection Matrix P:")
print(np.round(P, 4))

# (b) Project the 3D point into Homogeneous 2D Space
X_G_homogeneous = np.vstack((X_G, [[1]]))
x_img_homogeneous = P @ X_G_homogeneous

# Convert from homogeneous to Euclidean 2D pixel coordinates by dividing by Z
Z_depth = x_img_homogeneous[2, 0]
u_projected = x_img_homogeneous[0, 0] / Z_depth
v_projected = x_img_homogeneous[1, 0] / Z_depth

print("\n--- Question 1(b) ---")
print(f"Projected Image Coordinates (u, v): ({u_projected:.4f}, {v_projected:.4f})")
print(f"Computed Depth (Z): {Z_depth:.4f}")
print("Verification Note: Since Z is negative, the 3D point is physically behind the camera viewpoint.")

# (c) Calculate Euclidean Re-projection Error
reprojection_error = np.sqrt((u_projected - obs_u)**2 + (v_projected - obs_v)**2)

print("\n--- Question 1(c) ---")
print(f"Re-projection Error: {reprojection_error:.4f} pixels")

Question 2 Calibrate Camera Using ChArUco Pattern

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# ==========================================
# CONFIGURATION BLOCK
# ==========================================
# Directory containing your calibration photos
CALIBRATION_DIR = "calibration_images"

# Specifications for the target ChArUco pattern linked in the prompt
CHARUCO_SQUARES_X = 5
CHARUCO_SQUARES_Y = 7

# Measure your physical printed pattern with a ruler and input values in meters!
SQUARE_METERS = 0.040  # Example: 40 millimeters
MARKER_METERS = 0.020  # Example: 20 millimeters

def run_charuco_calibration(data_directory):
    """
    Reads images from directory, processes ChArUco intersections,
    renders a verification view, and computes camera intrinsics.
    """
    # Initialize the ArUco structural objects based on standard 6x6 configuration
    dictionary = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
    charuco_board = cv2.aruco.CharucoBoard(
        (CHARUCO_SQUARES_X, CHARUCO_SQUARES_Y),
        SQUARE_METERS,
        MARKER_METERS,
        dictionary
    )
    detector_params = cv2.aruco.DetectorParameters()

    # Storage arrays for accumulated calibration points
    all_corners = []
    all_ids = []
    frame_dimensions = None
    sample_render = None

    # Read files safely using pathlib
    dir_path = Path(data_directory)
    valid_extensions = {'.jpg', '.jpeg', '.png'}
    image_files = [f for f in dir_path.iterdir() if f.suffix.lower() in valid_extensions] if dir_path.exists() else []

    if not image_files:
        print(f"Framework Ready. Place your smartphone images inside the folder: '{data_directory}/'")
        return None, None

    print(f"Found {len(image_files)} calibration snapshots. Extracting features...")

    for file in image_files:
        frame = cv2.imread(str(file))
        if frame is None:
            continue

        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if frame_dimensions == None:
            frame_dimensions = gray_frame.shape[::-1]

        # Step 1: Detect base ArUco components within white cells
        marker_corners, marker_ids, _ = cv2.aruco.detectMarkers(
            gray_frame, dictionary, parameters=detector_params
        )

        # Step 2: Interpolate internal chessboard square intersections
        if marker_ids is not None and len(marker_ids) > 0:
            retval, c_corners, c_ids = cv2.aruco.interpolateCornersCharuco(
                marker_corners, marker_ids, gray_frame, charuco_board
            )

            # Require at least 4 stable features to avoid sub-pixel initialization noise
            if c_corners is not None and c_ids is not None and len(c_corners) > 3:
                all_corners.append(c_corners)
                all_ids.append(c_ids)

                # Cache the first successful match frame for homework submission snapshot
                if sample_render is None:
                    sample_render = frame.copy()
                    cv2.aruco.drawDetectedCornersCharuco(sample_render, c_corners, c_ids, (0, 255, 0))
                    sample_render = cv2.cvtColor(sample_render, cv2.COLOR_BGR2RGB)

    # Display the verification image inline (Requirement 2c)
    if sample_render is not None:
        plt.figure(figsize=(9, 7))
        plt.imshow(sample_render)
        plt.title("Requirement 2(c): Corner Verification Snapshot")
        plt.axis('off')
        plt.show()
    else:
        print("Error: ChArUco pattern could not be recognized. Ensure lighting is clear and checking boards match.")
        return None, None

    # Step 3: Run final optimization to solve for Intrinsics Matrix K
    print("Computing optimization parameters...")
    success, K_matrix, distortion_coefficients, _, _ = cv2.aruco.calibrateCameraCharuco(
        charucoCorners=all_corners,
        charucoIds=all_ids,
        board=charuco_board,
        imageSize=frame_dimensions,
        cameraMatrix=None,
        distCoeffs=None
    )

    print("\n--- Question 2(d): Calibration Results ---")
    print("Camera Intrinsic Matrix (K):")
    print(np.round(K_matrix, 2))
    print(f"\nExtracted Focal Length Parameters: fx = {K_matrix[0,0]:.2f} pixels, fy = {K_matrix[1,1]:.2f} pixels")
    print(f"Extracted Center Principal Point: cx = {K_matrix[0,2]:.2f} pixels, cy = {K_matrix[1,2]:.2f} pixels")

    return K_matrix, distortion_coefficients

# To run calibration once pictures are taken:
# K, dist = run_charuco_calibration(CALIBRATION_DIR)

--- Part 1: Projection of a 3D point ---

(a) Projection Matrix P:
[[ 2.57424000e+02 -4.05120000e+02  3.20000000e+02  9.22904094e+04]
 [ 4.05120000e+02  2.57424000e+02  2.70000000e+02  8.64248200e+03]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00 -4.00000000e+02]] 

(b) Projected Image Coordinates: (u, v) = (-626.3651, -176.1574)

(c) Re-projection Error: 933.9826 pixels

--- Part 2: Calibrate your own camera (Framework) ---

--- Part 3: SIFT Extraction and Matching (Framework) ---



Question 3
Image Feature Extraction and Matching

In [7]:
import cv2
import matplotlib.pyplot as plt

def compute_sift_correspondences(img1_path, img2_path):
    """
    Extracts SIFT features from two viewpoints and isolates regular inliers
    versus outliers using Lowe's ratio distance processing.
    """
    # Read incoming source files in grayscale mode
    image_a = cv2.imread(img1_path, cv2.IMREAD_GRAYSCALE)
    image_b = cv2.imread(img2_path, cv2.IMREAD_GRAYSCALE)

    if image_a is None or image_b is None:
        print("Missing assignment files. Please verify paths.")
        return

    # 1. Initialize SIFT Detector and compute descriptors
    sift_engine = cv2.SIFT_create()
    keypoints_a, descriptors_a = sift_engine.detectAndCompute(image_a, None)
    keypoints_b, descriptors_b = sift_engine.detectAndCompute(image_b, None)

    # Print out spatial specs for a single representative feature (Requirement 3b)
    if keypoints_a:
        example_kp = keypoints_a[0]
        print("--- Question 3(b) ---")
        print(f"Representative Keypoint Metric -> Pixel Scale Size: {example_kp.size:.2f}")
        print(f"Representative Keypoint Metric -> Dominant Angular Orientation: {example_kp.angle:.2f}°\n")

    # 2. Run raw Brute-Force matching with k=2 nearest entries
    matcher = cv2.BFMatcher()
    raw_knn_matches = matcher.knnMatch(descriptors_a, descriptors_b, k=2)

    # 3. Apply Lowe's Ratio Test to sort inliers from outliers
    inlier_matches = []
    outlier_matches = []

    for first_match, second_match in raw_knn_matches:
        if first_match.distance < 0.75 * second_match.distance:
            inlier_matches.append([first_match])
        else:
            outlier_matches.append([first_match])

    print("--- Question 3(c) ---")
    print(f"Total computed features evaluated: {len(raw_knn_matches)}")
    print(f"Distinct Inlier matches passing test: {len(inlier_matches)}")
    print(f"Rejected Outlier matches failing test: {len(outlier_matches)}")

    # 4. Generate Subplot Visualizations for Report Submission
    fig, axes = plt.subplots(2, 1, figsize=(14, 11))

    # Top Panel: Clean Green Lines for True Structural Inliers
    inlier_render = cv2.drawMatchesKnn(
        image_a, keypoints_a, image_b, keypoints_b, inlier_matches[:20], None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS, matchColor=(0, 255, 0)
    )
    axes[0].imshow(cv2.cvtColor(inlier_render, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Representative Inliers (Top 20 Unique Correspondences Passing Ratio Threshold)")
    axes[0].axis('off')

    # Bottom Panel: Crimson Lines indicating rejected Outlier matches
    outlier_render = cv2.drawMatchesKnn(
        image_a, keypoints_a, image_b, keypoints_b, outlier_matches[:20], None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS, matchColor=(255, 0, 0)
    )
    axes[1].imshow(cv2.cvtColor(outlier_render, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Representative Outliers (Top 20 Ambiguous/Rejected Matches)")
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# To execute feature pipeline:
# compute_sift_correspondences("my_face_left.png", "my_face_right.png")

(a) Camera Projection Matrix P:
[[ 2.57424000e+02 -4.05120000e+02  3.20000000e+02  9.22904094e+04]
 [ 4.05120000e+02  2.57424000e+02  2.70000000e+02  8.64248200e+03]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00 -4.00000000e+02]]

(b) Projected Image Coordinates (u, v): (-626.3651, -176.1574)

(c) Re-projection Error: 933.9826 pixels
